# 12 — Shaoxing inventory, leakage audit and cohort preparation

Validate all headers and screen exact waveform-file duplicates BEFORE random sampling. Default excludes all members of duplicate groups without using diagnosis labels. No source files are modified. Review leakage_audit.json before external inference. Cross-dataset and near-duplicate checks are not implied by within-dataset deduplication. Existing frozen cohorts require a new output directory. Restart the kernel after replacing the helper.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'config.json').is_file()), None)
if ROOT is None: raise FileNotFoundError('Start inside graph_reasoning')
sys.path.insert(0, str(ROOT))
from report_study import read, resolve
CONFIG = read(ROOT / 'config.json')
OUT = resolve(ROOT, CONFIG['output_dir'])
BASE_STUDY = OUT / 'report_study_v1'
EXTERNAL = OUT / 'shaoxing_external_v1'
print('Kernel:', sys.executable)
print('External cohort:', EXTERNAL)


In [ ]:
from shaoxing_external import prepare, CODE_MAP
DATASET_ROOT = Path('/path/to/data/physionet.org/files/ecg-arrhythmia/1.0.0')
# Local smoke-test only: ROOT / 'Shaoxing/ecg-arrhythmia/1.0.0'; use a separate EXTERNAL path.
MAX_RECORDS = 0  # All available records; use e.g. 1000 only as a prespecified seeded sample.
SEED = 43
REVIEWED_MAPPING = dict(CODE_MAP)
NEGATIVE_CONCEPTS = []  # Unlisted diagnosis != confirmed absence. Do not fill all labels with 0.
MAPPING_REVIEW_NOTE = 'draft; clinical review pending'
INVALID_HEADER_POLICY = 'exclude'  # Explicitly audit and exclude malformed selected records.
DUPLICATE_POLICY = 'exclude_all'  # Exclude every member of exact duplicate-file groups.
REFERENCE_HASHES_PATH = None  # Optional JSON inventory; see SHAOXING_LEAKAGE_CONTROL.txt.
print(prepare(DATASET_ROOT, EXTERNAL, limit=MAX_RECORDS, seed=SEED,
              reviewed_mapping=REVIEWED_MAPPING, negative_concepts=NEGATIVE_CONCEPTS,
              mapping_review_note=MAPPING_REVIEW_NOTE, invalid_header_policy=INVALID_HEADER_POLICY,
              duplicate_policy=DUPLICATE_POLICY, reference_hashes_path=REFERENCE_HASHES_PATH))
print(read(EXTERNAL / 'label_audit.json'))

print(read(EXTERNAL / 'header_validation_audit.json'))

leakage = read(EXTERNAL / 'leakage_audit.json')
print({k:v for k,v in leakage.items() if k not in ('duplicate_groups','reference_overlap_records')})
print('Full audit:', EXTERNAL / 'leakage_audit.json')
